In [34]:
import os
import time
import json
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

from datasets import load_dataset, Audio

In [35]:
dataset = load_dataset("yakhyo/mozilla-common-voice-uzbek", split="train+validation")
dataset

Dataset({
    features: ['client_id', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant', 'text'],
    num_rows: 60609
})

In [36]:
SAMPLE_RATE = 22050
dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))

In [37]:
df = dataset.remove_columns(["audio"]).to_pandas()
df.head()

,client_id,sentence,up_votes,down_votes,age,gender,accent,locale,segment,variant,text
0,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Bugun ertalab Gyotenikiga taklifnoma oldim.,2,0,twenties,male_masculine,,uz,,,Bugun ertalab Gyotenikiga taklifnoma oldim.
1,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Uning badiiy tasvir imkoniyatlarini rivojlanti...,2,0,twenties,male_masculine,,uz,,,Uning badiiy tasvir imkoniyatlarini rivojlanti...
2,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Udan ko’ra balandroq joy bor.,2,1,twenties,male_masculine,,uz,,,Udan ko'ra balandroq joy bor.
3,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Bu jumlada fig‘oni falakka chiqib birikmasi ib...,2,1,twenties,male_masculine,,uz,,,Bu jumlada fig'oni falakka chiqib birikmasi ib...
4,2160561702bac0e2048d2dc79810c2d8a6e6942a6dcac8...,Bundan tashqari puxta jamlangan kutubxona bor.,2,1,twenties,male_masculine,,uz,,,Bundan tashqari puxta jamlangan kutubxona bor.


In [38]:
new_df = df[["gender"]].copy()
new_df.insert(0, "idx", range(len(new_df)))
new_df

,idx,gender
0,0,male_masculine
1,1,male_masculine
2,2,male_masculine
3,3,male_masculine
4,4,male_masculine
...,...,...
60604,60604,male_masculine
60605,60605,male_masculine
60606,60606,male_masculine
60607,60607,male_masculine


In [39]:
MALE = "male_masculine"
FEMALE = "female_feminine"

new_df = new_df[(new_df["gender"] == MALE) | (new_df["gender"] == FEMALE)]
new_df.shape

(33653, 2)

In [40]:
male = new_df[new_df["gender"] == MALE]
female = new_df[new_df["gender"] == FEMALE]

print("male:  ", male.shape)
print("female:", female.shape)

male:   (21412, 2)
female: (12241, 2)


In [41]:
male = male.iloc[0:12000]

# videoda: female = female.append(male)
female = pd.concat([female, male])
female

,idx,gender
20118,20118,female_feminine
20119,20119,female_feminine
20120,20120,female_feminine
20121,20121,female_feminine
20122,20122,female_feminine
...,...,...
18741,18741,male_masculine
18742,18742,male_masculine
18743,18743,male_masculine
18744,18744,male_masculine


In [42]:
# shuffle the dataframe
female = female.sample(frac=1).reset_index(drop=True)
female

,idx,gender
0,44776,female_feminine
1,45749,female_feminine
2,11548,male_masculine
3,46014,female_feminine
4,45561,female_feminine
...,...,...
24236,55339,female_feminine
24237,15119,male_masculine
24238,10624,male_masculine
24239,11954,male_masculine


In [43]:
def load_audio(ref):
    """Videodagi librosa.load(path) ning o'rnini bosadi.
    ref - idx raqami yoki jadval qatori."""
    if isinstance(ref, pd.Series):
        ref = ref["idx"]
    samples = dataset[int(ref)]["audio"].get_all_samples()
    return samples.data.numpy().squeeze(), samples.sample_rate


X_audio, sample_rate = load_audio(female.iloc[0])
print("audio:", X_audio.shape, "| sr:", sample_rate)
print("gender:", female.iloc[0]["gender"])

audio: (39690,) | sr: 22050
gender: female_feminine


In [44]:
def extract_feature(ref):
    X, sample_rate = load_audio(ref)
    result = np.array([])
    mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
    result = np.hstack((result, mel))

    return result


extract_feature(female.iloc[0]).shape   # (128,)

(128,)

In [45]:
dirname = "results"

if not os.path.isdir(dirname):
    os.mkdir(dirname)

In [46]:
# save to csv file
female.to_csv("common_voice_uz.csv", index=False)

In [47]:
LABELS = {FEMALE: 0, MALE: 1}   # videoda: 0 = female, 1 = male


def extract_feature_batch(batch):
    out = []
    for a in batch["audio"]:
        s = a.get_all_samples()
        X = s.data.numpy().squeeze()
        mel = np.mean(librosa.feature.melspectrogram(y=X, sr=s.sample_rate).T, axis=0)
        out.append(mel)
    return {"features": out}


labeled = dataset.filter(lambda x: x["gender"] in LABELS)
labeled = labeled.map(lambda x: {"label": LABELS[x["gender"]]})

featured = labeled.map(
    extract_feature_batch,
    batched=True,
    batch_size=64,
    remove_columns=["audio"],
    num_proc=4,
)

np.save("results/features.npy", np.array(featured["features"]))
np.save("results/labels.npy", np.array(featured["label"]))
print("saqlandi")

Map (num_proc=4): 100%|██████████| 33653/33653 [25:45<00:00, 21.78 examples/s]


saqlandi


In [48]:
def load_data(vector_length=128):

    if not os.path.isdir("results"):
        print("Results directory not found, please run the preprocessing script first.")
        return None, None

    # if features & labels already loaded individually and bundled, load them from there instead
    if os.path.isfile("results/features.npy") and os.path.isfile("results/labels.npy"):
        X = np.load("results/features.npy")
        y = np.load("results/labels.npy")
        return X, y

In [49]:
X, y = load_data()

print("X:", X.shape)
print("y:", y.shape)
print("0 = female, 1 = male ->", np.bincount(y))

X: (33653, 128)
y: (33653,)
0 = female, 1 = male -> [12241 21412]


In [50]:
from sklearn.model_selection import train_test_split


def split_data(X, y, test_size=0.1, valid_size=0.1):
    # split training set and testing set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=7)
    # split training set and validation set
    X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=valid_size, random_state=7)
    # return a dictionary of values
    return {
        "X_train": X_train,
        "X_valid": X_valid,
        "X_test": X_test,
        "y_train": y_train,
        "y_valid": y_valid,
        "y_test": y_test
    }

In [51]:
data = split_data(X, y, test_size=0.1, valid_size=0.1)

for k, v in data.items():
    print(f"{k:8s} {v.shape}")

X_train  (27258, 128)
X_valid  (3029, 128)
X_test   (3366, 128)
y_train  (27258,)
y_valid  (3029,)
y_test   (3366,)


In [52]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Input

I0000 00:00:1787980964.656726  971506 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787980964.921555  971506 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787980967.885346  971506 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [53]:
model = Sequential()
model.add(Input(shape=(128,)))            # videoda: Dense(256, input_shape=(128,))
model.add(Dense(256))
model.add(Dropout(0.3))
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))
# one output neuron with sigmoid activation function, 0 means female, 1 means male
model.add(Dense(1, activation="sigmoid"))
# using binary crossentropy as it's male/female classification (binary)
model.compile(loss="binary_crossentropy", metrics=["accuracy"], optimizer="adam")
# print summary of the model
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 156,545 (611.50 KB)

 Trainable params: 156,545 (611.50 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
batch_size = 64
epochs = 100

In [55]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(mode="min", patience=5, restore_best_weights=True)

In [56]:
model.fit(data["X_train"], data["y_train"],
          epochs=epochs,
          batch_size=batch_size,
          validation_data=(data["X_valid"], data["y_valid"]),
          callbacks=[early_stopping])

Epoch 1/100
426/426 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9422 - loss: 0.2277 - val_accuracy: 0.9759 - val_loss: 0.0952
Epoch 2/100
426/426 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9733 - loss: 0.1132 - val_accuracy: 0.9759 - val_loss: 0.0707
Epoch 3/100
426/426 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9766 - loss: 0.0900 - val_accuracy: 0.9772 - val_loss: 0.0624
Epoch 4/100
426/426 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9797 - loss: 0.0720 - val_accuracy: 0.9802 - val_loss: 0.0609
Epoch 5/100
426/426 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9802 - loss: 0.0695 - val_accuracy: 0.9832 - val_loss: 0.0517
Epoch 6/100
426/426 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9831 - loss: 0.0576 - val_accuracy: 0.9835 - val_loss: 0.0623
Epoch 7/100
426/426 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9847 - loss: 0.0490 - val_accuracy: 0.9868 - val_loss: 0.0558
Epoch 8/100
426/426 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9844 - loss: 0.0526 - val_accu

In [57]:
loss, accuracy = model.evaluate(data["X_test"], data["y_test"], verbose=0)

print(f"Test loss:     {loss:.4f}")
print(f"Test accuracy: {accuracy*100:.2f}%")

Test loss:     0.0417
Test accuracy: 99.17%


Videoda: `model.save("results/model.h5")`

`.h5` eskirgan format, Keras 3 da `.keras` ishlatiladi.

In [58]:
model.save("results/model.keras")   # videoda: model.save("results/model.h5")

In [59]:
def extract_feature_from_file(file_name):
    """Diskdagi fayl uchun - gradio yuklagan audio shu yerga tushadi."""
    X, sample_rate = librosa.load(file_name)
    result = np.array([])
    mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
    result = np.hstack((result, mel))
    return result


def classify_gender(file1, file2):
    file = file1 if file1 else file2
    if file is None:
        return json.dumps({"error": "audio yuklanmadi"})

    # get audio length
    audio_length = librosa.get_duration(path=file)   # videoda: filename=file (eskirgan)

    start = time.time()
    features = extract_feature_from_file(file).reshape(1, -1)
    male_prob = float(model.predict(features, verbose=0)[0][0])
    female_prob = 1 - male_prob
    gender = "male" if male_prob > female_prob else "female"
    end = time.time()
    final_time = end - start

    result = {
        "gender": gender,
        "male_probability": f"{male_prob*100:.2f}%",
        "female_probability": f"{female_prob*100:.2f}%",
        "Time taken": f"{final_time:.2f} seconds",
        "Audio length": f"{audio_length:.2f} seconds",
    }

    return json.dumps(result, indent=2)

In [61]:
import gradio as gr

demo = gr.Interface(
    fn=classify_gender,
    inputs=[
        gr.Audio(sources=["upload"], type="filepath", label="Fayl yuklash"),
        gr.Audio(sources=["microphone"], type="filepath", label="Mikrofon"),
    ],
    outputs="json",
)
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://75c4eccae446b0c298.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
